# Prototype 1: PDF → Markdown ingestion

Parses every PDF in `data/raw/` with MinerU and saves one Markdown file per PDF in `data/processed/markdown/`.

Flow: **config → list PDFs → parse + export → verify every page made it.**

Re-running is safe: a PDF is skipped only if its Markdown already contains *every* page.

In [ ]:
import re
import time
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from mineru import DoclibClient
from mineru.doclib import ParseRequest
from mineru.doclib.types import DocContentExportRequest

# ---- config ----
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
MARKDOWN_DIR = PROJECT_ROOT / "data" / "processed" / "markdown"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

TIER = "standard"      # one tier everywhere: flash | basic | standard | advanced
PAGE_RANGE = "all"     # IMPORTANT: MinerU parses only pages 1-10 if you don't say "all"
POLL_SECONDS = 5
TIMEOUT_SECONDS = 60 * 60   # per PDF; big PDFs can take a while

MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

client = DoclibClient()

In [ ]:
pdf_files = sorted(RAW_DIR.glob("*.pdf"))
assert pdf_files, f"No PDFs found in {RAW_DIR}"

display(pd.DataFrame([
    {"filename": p.name, "size_mb": round(p.stat().st_size / 1024**2, 2)}
    for p in pdf_files
]))

## Helpers

- `pages_in_markdown` reads the `<!-- page X of N -->` markers MinerU writes, so we can tell if a file is complete.
- `parse_pdf` asks for **all** pages, waits (with a timeout), then exports the whole document in one call.

In [ ]:
PAGE_MARKER = re.compile(r"<!-- page (\d+) of (\d+) -->")


def pages_in_markdown(md_path):
    """Return (pages_found, total_pages) from MinerU page markers. (0, None) if unknown."""
    if not md_path.exists():
        return 0, None
    markers = PAGE_MARKER.findall(md_path.read_text(encoding="utf-8"))
    if not markers:
        return 0, None
    pages = {int(page) for page, _ in markers}
    total = int(markers[0][1])
    return len(pages), total


def is_complete(md_path):
    found, total = pages_in_markdown(md_path)
    return total is not None and found == total


def wait_for_parse(parse_id):
    deadline = time.time() + TIMEOUT_SECONDS
    parse = client.get_parse(parse_id)
    while parse.status in ("pending", "parsing"):
        if time.time() > deadline:
            raise TimeoutError(f"Parse {parse_id} still '{parse.status}' after {TIMEOUT_SECONDS}s")
        time.sleep(POLL_SECONDS)
        parse = client.get_parse(parse_id)
    if parse.status != "done":
        raise RuntimeError(f"Parse {parse_id} ended as {parse.status}: {parse.error_code} {parse.error_msg}")


def parse_pdf(pdf_path):
    md_path = MARKDOWN_DIR / f"{pdf_path.stem}.md"

    submit = client.ensure_parse(ParseRequest(
        path=str(pdf_path.resolve()),
        tier=TIER,
        page_range=PAGE_RANGE,
    ))
    for parse_id in submit.wait_parse_ids:
        wait_for_parse(parse_id)

    # Export the whole document straight to a .md file (no page-by-page loop needed)
    client.export_doc_content(
        submit.short_id,
        DocContentExportRequest(tier=TIER, page_range=PAGE_RANGE, output=str(md_path.resolve())),
    )

    found, total = pages_in_markdown(md_path)
    return {
        "document": pdf_path.name,
        "status": "success" if found == total else "incomplete",
        "pages": f"{found}/{total}",
        "characters": md_path.stat().st_size,
        "markdown_path": str(md_path),
    }

## Parse every PDF

Failures don't stop the loop; they're recorded in `artifacts/ingestion_manifest.json`.

In [ ]:
manifest = []

for pdf in tqdm(pdf_files, desc="Parsing PDFs"):
    md_path = MARKDOWN_DIR / f"{pdf.stem}.md"

    if is_complete(md_path):
        found, total = pages_in_markdown(md_path)
        manifest.append({"document": pdf.name, "status": "skipped_complete",
                         "pages": f"{found}/{total}", "markdown_path": str(md_path)})
        continue

    try:
        manifest.append(parse_pdf(pdf))
    except Exception as exc:
        manifest.append({"document": pdf.name, "status": "failed", "error": str(exc)})

    # save after every PDF so a crash doesn't lose progress
    (ARTIFACTS_DIR / "ingestion_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

display(pd.DataFrame(manifest))

## Verify: every raw PDF has a complete Markdown file

In [ ]:
report = []
for pdf in pdf_files:
    md_path = MARKDOWN_DIR / f"{pdf.stem}.md"
    found, total = pages_in_markdown(md_path)
    text = md_path.read_text(encoding="utf-8") if md_path.exists() else ""
    report.append({
        "pdf": pdf.name,
        "markdown_exists": md_path.exists(),
        "pages_found": found,
        "pages_total": total,
        "complete": total is not None and found == total,
        "words": len(text.split()),
        "headings": sum(line.lstrip().startswith("#") for line in text.splitlines()),
        "preview": " ".join(text.split())[:150],
    })

report_df = pd.DataFrame(report)
display(report_df)

missing = report_df.loc[~report_df["complete"], "pdf"].tolist()
assert not missing, f"Not fully parsed: {missing}. Re-run the parse cell (or check the manifest for errors)."
print(f"All {len(pdf_files)} PDFs fully parsed ✔")